In [1]:
# Cell 1 — Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


All libraries imported successfully
Pandas version: 2.2.2
NumPy version: 1.26.4


In [2]:
file_path = r"C:\Customer_Revenue_Intelligence\data\raw\online_retail_II.xlsx"

df_1 = pd.read_excel(file_path, sheet_name="Year 2009-2010", engine="openpyxl")
print(f"Sheet 1 loaded: {df_1.shape[0]:,} rows, {df_1.shape[1]} columns")

df_2 = pd.read_excel(file_path, sheet_name="Year 2010-2011", engine="openpyxl")
print(f"Sheet 2 loaded: {df_2.shape[0]:,} rows, {df_2.shape[1]} columns")

df = pd.concat([df_1, df_2], ignore_index=True)
print(f"Combined dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

Sheet 1 loaded: 525,461 rows, 8 columns
Sheet 2 loaded: 541,910 rows, 8 columns
Combined dataset: 1,067,371 rows, 8 columns


In [3]:
print("=== FIRST 3 ROWS ===")
print(df.head(3).to_string())
print("\n=== COLUMN NAMES & DATA TYPES ===")
print(df.dtypes)
print("\n=== BASIC SHAPE ===")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

=== FIRST 3 ROWS ===
  Invoice StockCode                          Description  Quantity         InvoiceDate  Price  Customer ID         Country
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00   6.95      13085.0  United Kingdom
1  489434    79323P                   PINK CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom
2  489434    79323W                  WHITE CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom

=== COLUMN NAMES & DATA TYPES ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

=== BASIC SHAPE ===
Rows: 1,067,371
Columns: 8


In [4]:
print("=== NULL VALUE AUDIT ===")
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': null_pct
})
print(null_df)
print(f"\nTotal rows: {len(df):,}")
print(f"Rows with NULL Customer ID: {df['Customer ID'].isnull().sum():,}")
print(f"Rows KEEPING after removing null Customer ID: {df['Customer ID'].notnull().sum():,}")

=== NULL VALUE AUDIT ===
             Null Count  Null %
Invoice               0    0.00
StockCode             0    0.00
Description        4382    0.41
Quantity              0    0.00
InvoiceDate           0    0.00
Price                 0    0.00
Customer ID      243007   22.77
Country               0    0.00

Total rows: 1,067,371
Rows with NULL Customer ID: 243,007
Rows KEEPING after removing null Customer ID: 824,364


In [6]:
print("=== CANCELLATION AUDIT ===")
cancellations = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Cancellation rows (Invoice starts with C): {len(cancellations):,}")
print(f"Cancellation % of total: {round(len(cancellations)/len(df)*100, 2)}%")

print("\n=== NEGATIVE QUANTITY AUDIT ===")
neg_qty = df[df['Quantity'] < 0]
print(f"Rows with negative Quantity: {len(neg_qty):,}")
print(f"Negative Quantity % of total: {round(len(neg_qty)/len(df)*100, 2)}%")

print("\n=== ZERO OR NEGATIVE PRICE AUDIT ===")
zero_price = df[df['Price'] <= 0]
print(f"Rows with zero or negative Price: {len(zero_price):,}")
print(f"Zero/negative Price % of total: {round(len(zero_price)/len(df)*100, 2)}%")

print("\n=== NEGATIVE QUANTITY SAMPLE ===")
print(df[df['Quantity'] < 0][['Invoice','Quantity','Price','Customer ID']].head(5).to_string())

=== CANCELLATION AUDIT ===
Cancellation rows (Invoice starts with C): 19,494
Cancellation % of total: 1.83%

=== NEGATIVE QUANTITY AUDIT ===
Rows with negative Quantity: 22,950
Negative Quantity % of total: 2.15%

=== ZERO OR NEGATIVE PRICE AUDIT ===
Rows with zero or negative Price: 6,207
Zero/negative Price % of total: 0.58%

=== NEGATIVE QUANTITY SAMPLE ===
     Invoice  Quantity  Price  Customer ID
178  C489449       -12   2.95      16321.0
179  C489449        -6   1.65      16321.0
180  C489449        -4   4.25      16321.0
181  C489449        -6   2.10      16321.0
182  C489449       -12   2.95      16321.0


In [7]:
print("=== NEGATIVE QUANTITY WITHOUT C-INVOICE ===")
suspicious = df[(df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))]
print(f"Negative quantity rows WITHOUT C-invoice: {len(suspicious):,}")
print(f"\nSample of suspicious rows:")
print(suspicious[['Invoice','StockCode','Description','Quantity','Price','Customer ID']].head(10).to_string())
print(f"\nUnique Invoice prefixes in suspicious rows:")
print(suspicious['Invoice'].astype(str).str[0].value_counts())

=== NEGATIVE QUANTITY WITHOUT C-INVOICE ===
Negative quantity rows WITHOUT C-invoice: 3,457

Sample of suspicious rows:
     Invoice StockCode      Description  Quantity  Price  Customer ID
263   489464     21733     85123a mixed       -96    0.0          NaN
283   489463     71477            short      -240    0.0          NaN
284   489467    85123A      21733 mixed      -192    0.0          NaN
470   489521     21646              NaN       -50    0.0          NaN
3114  489655     20683              NaN       -44    0.0          NaN
3162  489660     35956             lost     -1043    0.0          NaN
3168  489663    35605A          damages      -117    0.0          NaN
4296  489806     18010              NaN      -770    0.0          NaN
4538  489820     21133  invcd as 84879?      -720    0.0          NaN
4566  489821    85049G              NaN      -240    0.0          NaN

Unique Invoice prefixes in suspicious rows:
Invoice
5    2728
4     729
Name: count, dtype: int64


In [9]:
print("=== SAMPLE OF DUPLICATE ROWS ===")
first_dup = df[df.duplicated(keep=False)].copy()
first_dup['Invoice'] = first_dup['Invoice'].astype(str)
first_dup = first_dup.sort_values('Invoice').head(6)
print(first_dup.to_string())

=== SAMPLE OF DUPLICATE ROWS ===
    Invoice StockCode                       Description  Quantity         InvoiceDate  Price  Customer ID         Country
362  489517     21913    VINTAGE SEASIDE JIGSAW PUZZLES         1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom
394  489517     21912          VINTAGE SNAKES & LADDERS         1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom
391  489517     21491   SET OF THREE VINTAGE GIFT WRAPS         1 2009-12-01 11:34:00   1.95      16329.0  United Kingdom
390  489517    84951A   S/4 PISTACHIO LOVEBIRD COASTERS         1 2009-12-01 11:34:00   2.55      16329.0  United Kingdom
388  489517    84951A   S/4 PISTACHIO LOVEBIRD COASTERS         1 2009-12-01 11:34:00   2.55      16329.0  United Kingdom
386  489517     21821  GLITTER STAR GARLAND WITH BELLS          1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom


In [10]:
print("=== DATE RANGE AUDIT ===")
print(f"Earliest date: {df['InvoiceDate'].min()}")
print(f"Latest date:   {df['InvoiceDate'].max()}")
print(f"Date span: {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} days")

print("\n=== TRANSACTIONS PER YEAR ===")
df['Year'] = df['InvoiceDate'].dt.year
print(df['Year'].value_counts().sort_index())

print("\n=== TRANSACTIONS PER MONTH (sample - 2010 only) ===")
df_2010 = df[df['Year'] == 2010]
print(df_2010['InvoiceDate'].dt.month.value_counts().sort_index())

=== DATE RANGE AUDIT ===
Earliest date: 2009-12-01 07:45:00
Latest date:   2011-12-09 12:50:00
Date span: 738 days

=== TRANSACTIONS PER YEAR ===
Year
2009     45228
2010    522714
2011    499429
Name: count, dtype: int64

=== TRANSACTIONS PER MONTH (sample - 2010 only) ===
InvoiceDate
1     31555
2     29388
3     41511
4     34057
5     35323
6     39983
7     33383
8     33306
9     42091
10    59098
11    78015
12    65004
Name: count, dtype: int64


In [11]:
print("=== COUNTRY DISTRIBUTION ===")
country_counts = df['Country'].value_counts()
country_pct = (df['Country'].value_counts() / len(df) * 100).round(2)
country_df = pd.DataFrame({
    'Transaction Count': country_counts,
    'Percentage %': country_pct
}).head(10)
print(country_df.to_string())

print(f"\nTotal unique countries: {df['Country'].nunique()}")
print(f"\nUK transactions: {country_counts.get('United Kingdom', 0):,}")
print(f"Non-UK transactions: {len(df) - country_counts.get('United Kingdom', 0):,}")
print(f"UK % of total: {round(country_counts.get('United Kingdom', 0)/len(df)*100, 2)}%")

=== COUNTRY DISTRIBUTION ===
                Transaction Count  Percentage %
Country                                        
United Kingdom             981330         91.94
EIRE                        17866          1.67
Germany                     17624          1.65
France                      14330          1.34
Netherlands                  5140          0.48
Spain                        3811          0.36
Switzerland                  3189          0.30
Belgium                      3123          0.29
Portugal                     2620          0.25
Australia                    1913          0.18

Total unique countries: 43

UK transactions: 981,330
Non-UK transactions: 86,041
UK % of total: 91.94%


In [12]:
print("=" * 55)
print("DATA QUALITY AUDIT SUMMARY")
print("=" * 55)
print(f"Total raw rows:                    {len(df):>10,}")
print(f"Null Customer ID rows:             {df['Customer ID'].isnull().sum():>10,}  ({round(df['Customer ID'].isnull().sum()/len(df)*100,2)}%)")
print(f"Cancellation rows (C-invoice):     {len(df[df['Invoice'].astype(str).str.startswith('C')]):>10,}  ({round(len(df[df['Invoice'].astype(str).str.startswith('C')])/len(df)*100,2)}%)")
print(f"Stock adjustment rows:             {3457:>10,}  (0.32%)")
print(f"Zero/negative price rows:          {len(df[df['Price'] <= 0]):>10,}  ({round(len(df[df['Price'] <= 0])/len(df)*100,2)}%)")
print(f"Duplicate rows:                    {df.duplicated().sum():>10,}  ({round(df.duplicated().sum()/len(df)*100,2)}%)")
print(f"Null Description rows:             {df['Description'].isnull().sum():>10,}  ({round(df['Description'].isnull().sum()/len(df)*100,2)}%)")
print("-" * 55)
print(f"Usable rows (customer-level):      {df['Customer ID'].notnull().sum():>10,}")
print(f"Date range:                        2009-12-01 to 2011-12-09")
print(f"Total countries:                   {df['Country'].nunique():>10,}")
print(f"UK transactions:                   {981330:>10,}  (91.94%)")
print("=" * 55)
print("\nISSUES TO FIX IN CLEANING (Day 3):")
print("1. Remove 34,335 duplicate rows")
print("2. Remove 19,494 cancellation invoices (C-prefix)")
print("3. Remove 3,457 stock adjustment rows (Price=0, no Customer)")
print("4. Remove 6,207 zero/negative price rows")
print("5. Convert Customer ID from float to integer (after null removal)")
print("6. Engineer Revenue column: Quantity x Price")
print("7. Flag and separate null Customer ID rows from clean dataset")

DATA QUALITY AUDIT SUMMARY
Total raw rows:                     1,067,371
Null Customer ID rows:                243,007  (22.77%)
Cancellation rows (C-invoice):         19,494  (1.83%)
Stock adjustment rows:                  3,457  (0.32%)
Zero/negative price rows:               6,207  (0.58%)
Duplicate rows:                        34,335  (3.22%)
Null Description rows:                  4,382  (0.41%)
-------------------------------------------------------
Usable rows (customer-level):         824,364
Date range:                        2009-12-01 to 2011-12-09
Total countries:                           43
UK transactions:                      981,330  (91.94%)

ISSUES TO FIX IN CLEANING (Day 3):
1. Remove 34,335 duplicate rows
2. Remove 19,494 cancellation invoices (C-prefix)
3. Remove 3,457 stock adjustment rows (Price=0, no Customer)
4. Remove 6,207 zero/negative price rows
5. Convert Customer ID from float to integer (after null removal)
6. Engineer Revenue column: Quantity x Price
7